In [1]:
import geopandas as gpd
import pandas as pd
import vaex

In [2]:
ano = 2024

In [3]:
agg_atributos = {
        'Quantidade de Unidades':'sum',
        'Quantidade de Unidades Condominiais':'sum',
        'Tamanho Médio da Unidade Condominial':'mean',
        'Tamanho médio dos Terrenos':'mean',
        'Área Total dos lotes':'sum',
        'Área Total Ocupada':'sum',
        'Área Total Construída':'sum',
        'Valor Total dos Terrenos':'sum',
        'Valor Total das Construções':'sum',
        'CA médio':'mean',
        'TO médio':'mean',
        'CA médio em lotes condominiais':'mean',
        'TO médio em lotes condominiais':'mean',
        'CA médio em lotes não condominiais':'mean',
        'TO médio em lotes não condominiais':'mean',
        'Comprimento Médio da Testada':'mean',
        'Número médio de Pavimentos':'mean',
        'Fator de obsolecência médio':'mean',
        'Residencial vertical Baixo (m2)':'sum',
        'Residencial vertical Médio (m2)':'sum',
        'Residencial vertical Alto (m2)':'sum',
        'Residencial horizontal Baixo (m2)':'sum',
        'Residencial horizontal Médio (m2)':'sum',
        'Residencial horizontal Alto (m2)':'sum',
        'Comercial vertical Baixo (m2)':'sum',
        'Comercial vertical Médio (m2)':'sum',
        'Comercial vertical Alto (m2)':'sum',
        'Comercial horizontal Baixo (m2)':'sum',
        'Comercial horizontal Alto (m2)':'sum',
        'Comercial horizontal Médio (m2)':'sum',
        'Terreno (m2)':'sum',
        'Outros Usos (m2)':'sum',
        'Residencial vertical Baixo (qt)':'sum',
        'Residencial vertical Médio (qt)':'sum',
        'Residencial vertical Alto (qt)':'sum',
        'Residencial horizontal Baixo (qt)':'sum',
        'Residencial horizontal Médio (qt)':'sum',
        'Residencial horizontal Alto (qt)':'sum',
        'Comercial vertical Baixo (qt)':'sum',
        'Comercial vertical Médio (qt)':'sum',
        'Comercial vertical Alto (qt)':'sum',
        'Comercial horizontal Baixo (qt)':'sum',
        'Comercial horizontal Alto (qt)':'sum',
        'Comercial horizontal Médio (qt)':'sum',
        'Terreno (qt)':'sum',
        'Outros Usos (qt)':'sum'
}

In [4]:
gdf_distritos = gpd.read_file('data/SIRGAS_GPKG_distrito.gpkg')

In [5]:
for distrito in gdf_distritos.iterrows():
    print(distrito[1].ds_nome)
    distrito = gdf_distritos[gdf_distritos.ds_codigo == distrito[1].ds_codigo].iloc[0]
    path = f'lotes_agregados_por_ano/{ano}/SIRGAS_SHP_LOTES_{distrito.ds_codigo.rjust(2, "0")}_{distrito.ds_nome.replace(" ", "_")}_IPTU_{ano}.gpkg'
    gdf_lote = gpd.read_file(path).drop_duplicates(subset=['sqlc']).set_index('sqlc')
    df_iptu = vaex.open(f'data/por_distritos/IPTU-1995-{2024}-agrupados-por-sqlc-{distrito.ds_codigo}-{distrito.ds_nome.replace(" ", "-").lower()}.hdf5').to_pandas_df().set_index('sqlc')
    df_iptu = df_iptu[df_iptu.ano == ano]
    lotes_existentes = gdf_lote.join(df_iptu, how='inner')
    lotes_sg = df_iptu.join(gdf_lote, how='left').sq.isna()
    df_lotes_sg = df_iptu[lotes_sg].reset_index()
    df_lotes_sg.sqlc = df_lotes_sg.sqlc.str[:6] + '000000'
    df_lotes_sg_group = df_lotes_sg.groupby('sqlc').agg(agg_atributos)
    lotes_agregados = gdf_lote.join(df_lotes_sg_group, how='inner')
    lotes = pd.concat([lotes_existentes, lotes_agregados])
    lotes.to_file(f"iptu_por_lotes/{ano}/IPTU-SP-todos-atributos-por-lotes-{distrito.ds_codigo}-{distrito.ds_nome.lower().replace(' ', '-')}.gpkg", driver='GPKG')
    # break

PIRITUBA
SAO DOMINGOS
JARAGUA
BRASILANDIA
FREGUESIA DO O
CASA VERDE
CACHOEIRINHA
LIMAO
VILA GUILHERME
VILA MARIA
VILA MEDEIROS
ARTUR ALVIM
PENHA
CANGAIBA
VILA MATILDE
PONTE RASA
ERMELINO MATARAZZO
VILA CURUCA
ITAIM PAULISTA
GUAIANASES
LAJEADO
BARRA FUNDA
PERDIZES
VILA LEOPOLDINA
JAGUARA
LAPA
JAGUARE
REPUBLICA
SANTA CECILIA
SE
BELA VISTA
BOM RETIRO
CAMBUCI
CONSOLACAO
LIBERDADE
MOOCA
PARI
TATUAPE
AGUA RASA
BELEM
BRAS
CARRAO
VILA FORMOSA
ARICANDUVA
SAO MATEUS
SAO RAFAEL
IGUATEMI
VILA PRUDENTE
SAO LUCAS
MORUMBI
RIO PEQUENO
VILA SONIA
BUTANTA
RAPOSO TAVARES
PINHEIROS
ALTO DE PINHEIROS
ITAIM BIBI
JARDIM PAULISTA
CAMPO LIMPO
CAPAO REDONDO
VILA ANDRADE
JARDIM ANGELA
JARDIM SAO LUIS
SOCORRO
CIDADE DUTRA
GRAJAU
MARSILAC
PARELHEIROS
CIDADE TIRADENTES
PERUS
ANHANGUERA
SAPOPEMBA
SACOMA
CURSINO
IPIRANGA
MOEMA
SAUDE
VILA MARIANA
PEDREIRA
CIDADE ADEMAR
JACANA
TREMEMBE
MANDAQUI
SANTANA
TUCURUVI
SANTO AMARO
CAMPO GRANDE
CAMPO BELO
JABAQUARA
VILA JACUI
SAO MIGUEL
JARDIM HELENA
CIDADE LIDER
PARQUE DO CARM

In [6]:
df_iptu = df_iptu[df_iptu.ano == ano]
lotes_existentes = gdf_lote.join(df_iptu, how='inner')
lotes_sg = df_iptu.join(gdf_lote, how='left').sq.isna()
df_lotes_sg = df_iptu[lotes_sg].reset_index()
df_lotes_sg.sqlc = df_lotes_sg.sqlc.str[:6] + '000000'
df_lotes_sg_group = df_lotes_sg.groupby('sqlc').agg(agg_atributos)
lotes_agregados = gdf_lote.join(df_lotes_sg_group, how='inner')
lotes = pd.concat([lotes_existentes, lotes_agregados])

In [7]:
lotes

,sq,agregado,geometry,ano,Quantidade de Unidades,Quantidade de Unidades Condominiais,Tamanho Médio da Unidade Condominial,Tamanho médio dos Terrenos,Área Total dos lotes,Área Total Ocupada,...,Residencial horizontal Médio (qt),Residencial horizontal Alto (qt),Comercial vertical Baixo (qt),Comercial vertical Médio (qt),Comercial vertical Alto (qt),Comercial horizontal Baixo (qt),Comercial horizontal Alto (qt),Comercial horizontal Médio (qt),Terreno (qt),Outros Usos (qt)
sqlc,,,,,,,,,,,,,,,,,,,,,
138084001600,138084,False,"POLYGON ((353817.626 7396743.921, 353828.192 7...",2024.0,1,0,NaN,340.0,340.0,120.0,...,0,0,0,0,0,1,0,0,0,0
144064000800,144064,False,"POLYGON ((352135.392 7394825.776, 352114.098 7...",2024.0,1,0,NaN,3150.0,3150.0,700.0,...,0,0,0,0,0,1,0,0,0,0
114312017900,114312,False,"POLYGON ((350809.372 7395068.755, 350806.332 7...",2024.0,1,0,NaN,1571.0,1571.0,76.0,...,0,0,0,0,0,1,0,0,0,0
140321000300,140321,False,"POLYGON ((351231.913 7398461.209, 351251.124 7...",2024.0,1,0,NaN,147.0,147.0,62.0,...,0,0,0,0,0,0,0,0,0,0
140098002600,140098,False,"POLYGON ((351124.607 7398341.879, 351120.941 7...",2024.0,1,0,NaN,65.0,65.0,36.0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144205000000,144205,True,"POLYGON ((352220.593 7395464.027, 352225.263 7...",NaN,4,0,NaN,1423.5,5694.0,150.0,...,0,0,0,0,0,1,0,0,3,0
144209000000,144209,True,"POLYGON ((352510.789 7394433.116, 352511.403 7...",NaN,2,0,NaN,1853.0,3706.0,2983.0,...,0,0,0,0,0,1,0,0,0,0
230090000000,230090,True,None,NaN,1,0,NaN,287.0,287.0,0.0,...,0,0,0,0,0,0,0,0,1,0


In [8]:
df_iptu

,ano,Quantidade de Unidades,Quantidade de Unidades Condominiais,Tamanho Médio da Unidade Condominial,Tamanho médio dos Terrenos,Área Total dos lotes,Área Total Ocupada,Área Total Construída,Valor Total dos Terrenos,Valor Total das Construções,...,Residencial horizontal Médio (qt),Residencial horizontal Alto (qt),Comercial vertical Baixo (qt),Comercial vertical Médio (qt),Comercial vertical Alto (qt),Comercial horizontal Baixo (qt),Comercial horizontal Alto (qt),Comercial horizontal Médio (qt),Terreno (qt),Outros Usos (qt)
sqlc,,,,,,,,,,,,,,,,,,,,,
114022002900,2024,1,0,NaN,240.0,240.000000,50.000000,50,2.880000e+05,63700.0,...,0,0,0,0,0,0,0,0,0,0
114029002900,2024,1,0,NaN,180.0,180.000000,50.000000,50,2.259000e+05,63700.0,...,0,0,0,0,0,0,0,0,0,0
114033000400,2024,1,0,NaN,500.0,500.000000,361.000000,362,5.235000e+05,532140.0,...,0,0,0,0,0,1,0,0,0,0
114039000800,2024,1,0,NaN,137.0,137.000000,60.000000,60,1.189160e+05,76440.0,...,0,0,0,0,0,0,0,0,0,0
114052002300,2024,1,0,NaN,125.0,125.000000,49.000000,49,1.172500e+05,74529.0,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138063016700,2024,1,0,NaN,200.0,200.000000,91.000000,149,1.070000e+05,226629.0,...,1,0,0,0,0,0,0,0,0,0
140023000001,2024,10,10,48.8,297.0,297.000002,175.000001,488,2.162160e+05,881816.0,...,0,0,0,0,0,0,0,0,0,0
138054007800,2024,1,0,NaN,91.0,91.000000,49.000000,90,4.977700e+04,127800.0,...,0,0,0,0,0,0,0,0,0,0
